##### Import the libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
import re
import spacy
import nltk
from nltk.corpus import stopwords
import plotly.express as px

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\balog\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\balog\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

##### Load the dataset

In [2]:
df = pd.read_csv(r'C:\Users\balog\OneDrive\Documents\Olayemi\Intership\Customer Feedback Analysis\Data\reviews.csv')


##### Data Cleaning

- Convert timestamp to datetime

In [3]:
df_clean = df.copy()

In [4]:
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"], errors="coerce")
df_clean["timestamp"].dtype

datetime64[ns, UTC]

- Check the missing value under 'country'

In [5]:
df_clean[df_clean["country"].isna()]

,review_id,product_category,timestamp,country,rating,review,sentiment
9380,REV-FC39A09F,Sports,2021-12-30 21:22:01+00:00,NaN,5,Have purchased three separate items and had th...,Positive


- Drop the row with the missing value under 'country'

In [6]:
df_clean.dropna(subset=["country"], inplace=True)

- Identify duplicate reviews

In [7]:
df_clean["review"].duplicated().sum()

np.int64(648)

- Check the duplicated review across other features

In [8]:
df_clean.groupby("review").agg({
    "product_category": "nunique",
    "country": "nunique",
    "rating": "nunique",
    "sentiment": "nunique"
}).sort_values(by="country",ascending=False).head(20)

,product_category,country,rating,sentiment
review,,,,
Review text not found,7,60,5,3
"Who would choose to sell on Amazon, what a rip off. For a long time any items being returned we could deduct return fees if someone changed there mind or made a mistake. We get charged to send to customer and full price to return no choice.I question why o why did i sell on Amazon. I am looking for other ways to not be stolen from. Let alone all the other bad service and extra extra fees.",1,2,1,1
Great place to get quality products at a great price and with fast shipping,2,2,1,1
I was disappointed with my purchase. The quality of the product was not what I expected and it did not live up to the description provided. I think it is overpriced for what it delivers.,1,2,1,1
Waited 5 days to process my payment that by then didn't go through then waited a week at midnight when the payment was in there and cancelled my order! I will never order anything from Amazon again!!!!!!!,2,2,1,1
Worst experience on online shopping ever!! Amazon can’t keep track on what their sellers mail to their buyers. I ordered the MPow headphones (red color) and instead received a black/neon green pair of headphones made by Yuanguo. I tried to get my money back and instead their costumer services negotiated with me to resend a replacement and Amazon costumer services guaranteed that this time I would receive what I bought. The replacement arrived and now I’m stuck with 4 pairs on earbuds I never ordered. I’m very upset and would like to be sent the product that I actually ordered. I will never buy anything with amazon again. Amazon needs to wake up for that and keep a better track on what’s being send by the sellers.,2,2,2,2
I hate that I order from amazon for over 7 yrs and now they giving me hell because I didn’t receive my product as stated so I had to go and purchase the items because I had to do a wedding parties hair. Now they won’t refund my money I been waiting since Oct 11th,1,2,1,1
SRAIGHT AS A RULER HELP FUL RELIABLE ADMIT MISTAKE- IF A REFUND IS DUE YOU RECEIV IT ALMOSTIMMEDIATELYNOT LIKE CESDEALS THAT IS APARADISE FOR CROOKS,2,2,1,1
good company,2,2,2,1


     Many review texts are repeated across multiple countries and product categories. The review "Review text not found" appeared across 60 countries with varying ratings and sentiment labels, so all reviews with this statement would be removed. Other duplicated reviews to be remove as well.

- Remove "Review text not found"

In [9]:
df_clean[df_clean["review"] == "Review text not found"].head(10)

,review_id,product_category,timestamp,country,rating,review,sentiment
176,REV-ACB9FB65,Food & Grocery,2024-09-13 12:59:05+00:00,GB,1,Review text not found,Negative
184,REV-4351F6B9,Sports,2024-09-08 13:35:25+00:00,IN,1,Review text not found,Negative
188,REV-CA1B1E86,Electronics,2024-09-12 21:12:44+00:00,US,2,Review text not found,Negative
196,REV-85D1F564,Toys,2024-09-08 03:49:34+00:00,US,1,Review text not found,Negative
198,REV-8E15AA42,Sports,2024-09-05 13:35:52+00:00,US,4,Review text not found,Positive
199,REV-C3074BEC,Fashion,2024-09-06 12:10:56+00:00,GB,4,Review text not found,Positive
209,REV-635D16F3,Sports,2024-09-03 01:12:27+00:00,US,1,Review text not found,Negative
338,REV-5AC22C7E,Beauty,2024-08-23 13:06:49+00:00,US,4,Review text not found,Positive
339,REV-99B5C292,Fashion,2024-08-23 12:36:36+00:00,GB,4,Review text not found,Positive
347,REV-1F2E3558,Electronics,2024-08-22 15:14:07+00:00,US,4,Review text not found,Positive


In [10]:
rows_to_drop = df_clean[df_clean["review"] == "Review text not found"].index

In [11]:
df_clean = df_clean.drop(rows_to_drop)

In [12]:
df_clean[df_clean["review"] == "Review text not found"]

,review_id,product_category,timestamp,country,rating,review,sentiment


- Remove all other duplicated review

In [13]:
df_clean["review"].duplicated().sum()

np.int64(19)

In [14]:
df_clean = df_clean.drop_duplicates(subset="review")

In [15]:
df_clean["review"].duplicated().sum()

np.int64(0)

##### Language distribution within customer reviews.

- Detect the Language of Each Review

In [16]:
def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

In [17]:
df_clean["language"] = df_clean["review"].apply(detect_language)

- Validate Detection Samples

In [18]:
english = df_clean[df_clean["language"] == "en"][["review", "language"]].head(5)
english

,review,language
0,"I registered on the website, tried to order a ...",en
1,Had multiple orders one turned up and driver h...,en
2,I informed these reprobates that I WOULD NOT B...,en
3,I have bought from Amazon before and no proble...,en
4,If I could give a lower rate I would! I cancel...,en


- Calculate language proportions

In [19]:
language_summary = pd.DataFrame({
    "LanguageCount": df_clean["language"].value_counts(),
    "Language Percentage": round(df_clean["language"].value_counts(normalize=True) * 100, 2)
})

language_summary

,LanguageCount,Language Percentage
language,,
en,20225,99.12
it,26,0.13
fr,18,0.09
da,16,0.08
nl,14,0.07
de,12,0.06
af,11,0.05
ro,11,0.05
no,11,0.05


     Language detection revealed that customer reviews are primarily written in English (99.11%), while all other detected languages collectively represented less than 1% of the dataset. Therefore, advanced NLP preprocessing (stopword removal and lemmatization) would primarily be applied to English reviews while retaining all reviews in the dataset.

##### NLP preprocessing workflow

- Lowercase text on 'reviews'

In [20]:
df_clean["processed_review"] = df_clean["review"].str.lower()

- Remove URLs

In [21]:
df_clean["processed_review"] = df_clean["processed_review"].apply(lambda x: re.sub(r"http\S+|www\S+", "", str(x)))

- Remove emojis

In [22]:
df_clean["processed_review"] = df_clean["processed_review"].apply(lambda x: re.sub(r"[\U00010000-\U0010ffff]","",str(x),flags=re.UNICODE))

-  Remove punctations

In [23]:
df_clean["processed_review"] = df_clean["processed_review"].apply(lambda x: re.sub(r"[^\w\s]", "", str(x)))

- Remove extra spaces

In [24]:
df_clean["processed_review"] = (df_clean["processed_review"].str.replace(r"\s+", " ", regex=True).str.strip())

- Verify emojis and URLs are gone

In [25]:
# URLs
df_clean["processed_review"].str.contains(r"http\S+|www\S+", regex=True, na=False).sum()

# Emojis
df_clean["processed_review"].str.contains(r"[\U00010000-\U0010ffff]", regex=True, na=False).sum()

np.int64(0)

In [26]:
df_clean[["review", "processed_review"]].head(10)

,review,processed_review
0,"I registered on the website, tried to order a ...",i registered on the website tried to order a l...
1,Had multiple orders one turned up and driver h...,had multiple orders one turned up and driver h...
2,I informed these reprobates that I WOULD NOT B...,i informed these reprobates that i would not b...
3,I have bought from Amazon before and no proble...,i have bought from amazon before and no proble...
4,If I could give a lower rate I would! I cancel...,if i could give a lower rate i would i cancell...
5,Terrible you get customer service reps that ar...,terrible you get customer service reps that ar...
6,Amazon has a way of tainting a great product d...,amazon has a way of tainting a great product d...
7,I love amazon! I use it for half my shopping. ...,i love amazon i use it for half my shopping th...
8,I applied for a job with Amazon. I completed a...,i applied for a job with amazon i completed al...
9,I had a great experience with their customer s...,i had a great experience with their customer s...


- Remove stopwords (English only)

In [27]:
# Create stopword sets for english language only
stop_en = set(stopwords.words("english"))

# Function to remove stopwords
def remove_stopwords(text):

    words = str(text).split()

    cleaned_words = []

    for word in words:
        if word.lower() not in stop_en:
            cleaned_words.append(word)

    return " ".join(cleaned_words)

# Apply the remove_stopwords function only to English review
df_clean.loc[df_clean["language"] == "en", "processed_review"] = df_clean.loc[df_clean["language"] == "en", "processed_review"].apply(remove_stopwords)

In [28]:
df_clean[["review", "processed_review", "language"]].head(10)

,review,processed_review,language
0,"I registered on the website, tried to order a ...",registered website tried order laptop entered ...,en
1,Had multiple orders one turned up and driver h...,multiple orders one turned driver phone door n...,en
2,I informed these reprobates that I WOULD NOT B...,informed reprobates would going visit sick rel...,en
3,I have bought from Amazon before and no proble...,bought amazon problems happy service price ama...,en
4,If I could give a lower rate I would! I cancel...,could give lower rate would cancelled amazon p...,en
5,Terrible you get customer service reps that ar...,terrible get customer service reps clearly hom...,en
6,Amazon has a way of tainting a great product d...,amazon way tainting great product due inabilit...,en
7,I love amazon! I use it for half my shopping. ...,love amazon use half shopping prime membership...,en
8,I applied for a job with Amazon. I completed a...,applied job amazon completed steps including s...,en
9,I had a great experience with their customer s...,great experience customer service delivered or...,en


- Lemmatization of the 4 languages 

In [29]:
#Load language model

nlp_en = spacy.load("en_core_web_sm")

#lemmatization function
def lemmatize_text(text):

    doc = nlp_en(str(text))

    lemmas = [token.lemma_ for token in doc if token.is_alpha]

    return " ".join(lemmas)

# Apply lemmatization to the english language processed reviews
df_clean.loc[df_clean["language"] == "en", "processed_review"] = df_clean.loc[df_clean["language"] == "en", "processed_review"].apply(lemmatize_text)


In [30]:
df_clean[["review", "processed_review", "language"]].head(10)

,review,processed_review,language
0,"I registered on the website, tried to order a ...",register website try order laptop enter detail...,en
1,Had multiple orders one turned up and driver h...,multiple order one turn driver phone door numb...,en
2,I informed these reprobates that I WOULD NOT B...,inform reprobate would go visit sick relative ...,en
3,I have bought from Amazon before and no proble...,buy amazon problem happy service price amazon ...,en
4,If I could give a lower rate I would! I cancel...,could give low rate would cancel amazon prime ...,en
5,Terrible you get customer service reps that ar...,terrible get customer service rep clearly home...,en
6,Amazon has a way of tainting a great product d...,amazon way taint great product due inability s...,en
7,I love amazon! I use it for half my shopping. ...,love amazon use half shop prime membership wor...,en
8,I applied for a job with Amazon. I completed a...,applied job amazon complete step include send ...,en
9,I had a great experience with their customer s...,great experience customer service deliver orde...,en


In [31]:
print(df_clean.columns)

Index(['review_id', 'product_category', 'timestamp', 'country', 'rating',
       'review', 'sentiment', 'language', 'processed_review'],
      dtype='object')


##### Descriptive analysis after cleaning

In [32]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20405 entries, 0 to 21054
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   review_id         20405 non-null  object             
 1   product_category  20405 non-null  object             
 2   timestamp         20157 non-null  datetime64[ns, UTC]
 3   country           20405 non-null  object             
 4   rating            20405 non-null  int64              
 5   review            20405 non-null  object             
 6   sentiment         20405 non-null  object             
 7   language          20405 non-null  object             
 8   processed_review  20405 non-null  object             
dtypes: datetime64[ns, UTC](1), int64(1), object(7)
memory usage: 1.6+ MB


In [33]:
df_clean.describe()

,rating
count,20405.000000
mean,2.144621
std,1.659660
min,1.000000
25%,1.000000
50%,1.000000
75%,4.000000
max,5.000000


In [34]:
df_clean.describe(include=["object", "string", "bool"])

,review_id,product_category,country,review,sentiment,language,processed_review
count,20405,20405,20405,20405,20405,20405,20405
unique,20405,7,145,20405,3,26,20370
top,REV-50BCBCD9,Sports,US,"I registered on the website, tried to order a ...",Negative,en,good service
freq,1,3060,9095,1,14156,20225,5


     OVERVIEW
     The dataset contains 20405 customer reviews. 
     The average rating is 2.14 out of 5, and most reviews are 1-star, showing that customer feedback is mainly negative. 
     The dataset includes 7 product categories, with Sports receiving the highest number of reviews. 
     Reviews come from 145 countries, but the United States contributes the largest share.

In [35]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20405 entries, 0 to 21054
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   review_id         20405 non-null  object             
 1   product_category  20405 non-null  object             
 2   timestamp         20157 non-null  datetime64[ns, UTC]
 3   country           20405 non-null  object             
 4   rating            20405 non-null  int64              
 5   review            20405 non-null  object             
 6   sentiment         20405 non-null  object             
 7   language          20405 non-null  object             
 8   processed_review  20405 non-null  object             
dtypes: datetime64[ns, UTC](1), int64(1), object(7)
memory usage: 1.6+ MB


- Save the cleaned data

In [36]:
df_clean.to_csv("cleaned_reviews.csv")